# PROJECT SENTINEL — Portfolio Backtest Report
This notebook evaluates all currently saved and valid models against the Out-of-Sample (TEST2) data window.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Setup project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path: sys.path.append(PROJECT_ROOT)

import config
from utils.data_utils import load_snapshot_csv
from utils.model_utils import load_model, evaluate_model, list_valid_models
from utils.features import build_all_features_incremental

import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

## 1. Data Preparation
Loading the Test2 window features for all valid models.

In [ ]:
valid_tickers = list_valid_models()
print(f"Found {len(valid_tickers)} valid models: {valid_tickers}")

results = []
portfolio_equity = pd.Series(dtype=float)

for ticker in valid_tickers:
    # Load model and meta
    model, meta = load_model(ticker)
    if not model: continue
    
    # Load features from parquet
    from utils.features import _load_feature_parquet
    feat_df = _load_feature_parquet(ticker)
    if feat_df is None: continue
    
    # Determine TEST2 split boundary
    test2_days_start, test2_days_end = config.TEST2
    cutoff_start = datetime.utcnow() - timedelta(days=test2_days_start)
    cutoff_end = datetime.utcnow() - timedelta(days=test2_days_end)
    
    # Filter features for TEST2 window
    feat_df['timestamp'] = pd.to_datetime(feat_df['timestamp'])
    mask = (feat_df['timestamp'] >= cutoff_start) & (feat_df['timestamp'] <= cutoff_end)
    test_df = feat_df.loc[mask].copy()
    
    if test_df.empty: 
        print(f"[!] {ticker}: No data found for TEST2 window.")
        continue

    # Prepare X and y for evaluation
    # We need to recreate the label logic temporarily or rely on stored PF
    # To be accurate, we re-run evaluate_model using meta params
    X = test_df.drop(columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'], errors='ignore')
    
    # Re-calculate labels for this window to check accuracy
    from utils.labeling import generate_labels
    labels, atr_norm = generate_labels(
        test_df, 
        tp_pct=meta['tp_pct'], 
        sl_pct=meta['sl_pct'], 
        k1=meta['k1'], 
        k2=meta['k2']
    )
    
    # Evaluate
    metrics = evaluate_model(
        model, X, labels, 
        tp_pct=meta['tp_pct'], 
        sl_pct=meta['sl_pct'], 
        k1=meta['k1'], 
        k2=meta['k2'],
        atr_norm=atr_norm
    )
    
    # Calculate Dollar PnL
    # Risk-based sizing: Position Size = MAX_LOSS_USDT / sl_pct
    # PnL $ = Position Size * Expectancy (return per trade)
    avg_sl = meta['sl_pct'] + (meta['k2'] * atr_norm.mean() if atr_norm is not None else 0)
    pos_size_fixed_risk = config.MAX_LOSS_USDT / avg_sl
    pnl_dollars = metrics['trade_count'] * metrics['expectancy'] * pos_size_fixed_risk
    
    metrics.update({
        'ticker': ticker, 
        'pnl_usd': pnl_dollars, 
        'tp': meta['tp_pct'], 
        'sl': meta['sl_pct']
    })
    results.append(metrics)
    print(f"Evaluated {ticker:15} | Trades: {metrics['trade_count']:3} | PF: {metrics['PF']:.2f} | PnL: ${pnl_dollars:6.2f}")

## 2. Performance Summary Table

In [ ]:
report_df = pd.DataFrame(results)
if not report_df.empty:
    report_df = report_df[['ticker', 'trade_count', 'win_rate', 'PF', 'max_drawdown', 'expectancy', 'pnl_usd']]
    report_df.columns = ['Ticker', 'Trades', 'WinRate', 'PF', 'MaxDD', 'Expectancy', 'PnL ($)']
    
    total_pnl = report_df['PnL ($)'].sum()
    avg_winrate = report_df['WinRate'].mean()
    
    print("\n--- PORTFOLIO SUMMARY ---")
    print(f"Total Tickers: {len(report_df)}")
    print(f"Total Trades:  {report_df['Trades'].sum()}")
    print(f"Total PnL:     ${total_pnl:.2f}")
    print(f"Avg Win Rate:  {avg_winrate*100:.2f}%")
    
    display(report_df.sort_values('PnL ($)', ascending=False).style.background_gradient(subset=['PnL ($)'], cmap='RdYlGn'))
else:
    print("No valid models found to backtest.")

## 3. PnL Visualization

In [ ]:
if not report_df.empty:
    plt.figure(figsize=(12, 6))
    plt.bar(report_df['Ticker'], report_df['PnL ($)'], color=(report_df['PnL ($)'] > 0).map({True: 'g', False: 'r'}))
    plt.title('TEST2 Window Profit by Asset ($)')
    plt.ylabel('USDT Profit (Based on $5 Risk Per Trade)')
    plt.xticks(rotation=45)
    plt.axhline(0, color='black', linewidth=0.8)
    plt.show()